In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import fitsio
from pycorr import TwoPointCorrelationFunction, TwoPointEstimator, project_to_multipoles, project_to_wp, utils, setup_logging
from scipy.optimize import curve_fit
from LSS.common_tools import mknz, dl
from astropy.table import Table
import itertools

from dataloc import *
from groupcatalog import read_wp_file
from pyutils import get_max_observable_z
from nersc_clustering_tools import *

# MAKE ALL PLOTS TEXT BIGGER
plt.rcParams.update({'font.size': 15})
# But legend a bit smaller
plt.rcParams.update({'legend.fontsize': 12})
# Set DPI up a bit
plt.rcParams.update({'figure.dpi': 150})


In [ ]:
# First, figure out what BGS mag limit is equiv to SDSS maglimit due to photometry differences.
maglims = [-22.405, -21.815, -21.21, -20.64, -19.52] # These are the adjusted mag limits to match the number density of SDSS
zmaxes = [0.245169, 0.198666, 0.158833, 0.132333, 0.084833] # Fix volume to match SDSS
zmins = [0.02, 0.02, 0.02, 0.02, 0.02]
bgs_footprint = 7472 
sdss_footprint = 7700 # SDSS Footprint in Zehavi 2011 is 7700 deg^2
sdss_mags = [-22.0, -21.5, -21.0, -20.5, -19.5] # SDSS Zehavi 2011 magnitude thresholds
sdss_counts = [11385, 39456, 83238, 132225, 132664] # Number of galaxies in Zehavi 2011 for each threshold sample

# I don't know SDSS's exact n(z) so let's just do the overall density.
def galdensity(zmin, zmax, footprint, galcount):
    # Use DESI's dl method for luminosity distance to get the physical volume between zmin and zmax, then divide galcount by that volume to get density
    vol = (footprint/41253) * 4/3 * np.pi * (dl(zmax)**3 - dl(zmin)**3)
    return galcount / vol

# Function to integrate the two across a z range
def galdensity_nz(zmin, zmax):
    # Find idx for zmin and zmax
    idx1 = np.argmin(np.abs(NGC_nz[:,0] - zmin))
    idx2 = np.argmin(np.abs(NGC_nz[:,0] - zmax))

    # NGC
    zwidth = NGC_nz[:,2] - NGC_nz[:,1]
    integrated = NGC_nz[:,3] * zwidth 
    summed = integrated[idx1:idx2].sum()

    # And now for SGC
    idx1 = np.argmin(np.abs(SGC_nz[:,0] - zmin))
    idx2 = np.argmin(np.abs(SGC_nz[:,0] - zmax))
    zwidth = SGC_nz[:,2] - SGC_nz[:,1]
    integrated = SGC_nz[:,3] * zwidth
    summed += integrated[idx1:idx2].sum()

    return summed

# Make a luminosity threshold sample like SDSS to see if n(z) is same
def make_sdsslike_cuts(fcd, maglim, zmax, zmin):
    writename = fcd.replace('.dat.fits', f'_testcut.dat.fits')  
    arr = fitsio.read(fcd)
    arr = arr[arr['Z'] < zmax]
    arr = arr[arr['Z'] > zmin]
    arr = arr[arr['ABSMAG_R'] < maglim] # I *think* this is ABSMAG01_SDSS_R, my prep-extra-cols.py says so...
    fitsio.write(writename, arr, clobber=True)

# Function to combine the two to get an overall density, appropiatly weighting each by volume
def combine_nz(nz1, nz2):
    combined = np.zeros((len(nz1), 6))
    combined[:,0] = nz1[:,0]
    combined[:,1] = nz1[:,1]
    combined[:,2] = nz1[:,2]
    combined[:,4] = nz1[:,4] + nz2[:,4]
    combined[:,5] = nz1[:,5] + nz2[:,5]
    combined[:,3] = combined[:,4] / combined[:,5]
    return combined

for maglim, zmax, zmin, sdss_maglim, sdss_count in zip(maglims, zmaxes, zmins, sdss_mags, sdss_counts):
    #if zmax != 0.132333:
    #    continue
    outpath = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_FOOTPRINT.txt'
    outpath2 = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_FOOTPRINT.txt'

    make_sdsslike_cuts('/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_clustering.dat.fits', maglim, zmax, zmin)
    make_sdsslike_cuts('/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_clustering.dat.fits', maglim, zmax, zmin)

    cat = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_clustering_testcut.dat.fits'
    ran = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_NGC_0_clustering.ran.fits'
    mknz(cat, ran, outpath, zmax=0.3)

    cat = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_clustering_testcut.dat.fits'
    ran = '/global/cfs/cdirs/desi/users/ianw89/clustering/Y1/LSS/iron/LSScats/v1.5pip/BGS_BRIGHT_SGC_0_clustering.ran.fits'
    mknz(cat, ran, outpath2, zmax=0.3)

    # zmid zlow zhigh n(z) Nbin Vol_bin
    NGC_nz = np.loadtxt(outpath)
    SGC_nz = np.loadtxt(outpath2)

    total_nz = combine_nz(NGC_nz, SGC_nz)
    galcount = total_nz[:,4].sum()
    bgs_density = galdensity(0.02, zmax, bgs_footprint,  galcount)
    sdss_density = galdensity(0.02, zmax, sdss_footprint, sdss_count)

    print(f"BGS BRIGHT Y1 number of galaxies with M_r < {maglim} and z < {zmax}: {galcount:,.0f}. Density = {bgs_density :.4e} Mpc^-3")
    print(f"SDSS Zehavi 2011 number of galaxies with M_r < {sdss_maglim} and z < {zmax} :  {sdss_count:,}. Density = {sdss_density :.4e} Mpc^-3")
    print(f"Difference (%): {(bgs_density - sdss_density) / sdss_density * 100 :.2f}%")


In [ ]:
# Print off what the zmax should be if you want to use the 19.5 BGS flux limit for those magnitudes.
for maglim in maglims:
    zmax = get_max_observable_z(maglim, 19.5)
    print(f"For M_r < {maglim}, the zmax corresponding to the BGS flux limit of r=19.5 is {zmax:.4f}")

In [ ]:
# TODO Gotta load my rip of Zehavi like in the OLD clustering notebook. Compare clustering

clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/newclustering/Y1/LSS/iron/LSScats/v1.5pip/SDSS_COMPARE/'

In [ ]:
# Find all *_linclusimsysfit.png files in the clustering_base_dir and its subdirectories
import glob
linclusimsysfit_files = glob.glob(os.path.join(clustering_base_dir, '**', '*_linclusimsysfit.png'), recursive=True) 
print(f"Found {len(linclusimsysfit_files)} linclusimsysfit.png files:")
for f in linclusimsysfit_files:
    print(f" - {f}")

# First linclusimsysfit_files investigation
for i in range(10):
    first_file = linclusimsysfit_files[i]
    plt.figure(figsize=(10, 10))
    img = plt.imread(first_file)
    plt.imshow(img)
    plt.axis('off') 
    plt.show()

In [ ]:
# Example usage:
# Define the base directory where your clustering results are stored.
all_results = load_allcounts_from_disk(clustering_base_dir)

# You can now access the data and parameters like this:
if all_results:
    print("\nExample of first loaded result:")
    first_result = all_results[0]
    print("Parameters:", first_result['params'])
    print("Data object:", first_result['data'])



In [ ]:
# Now, call the function with the loaded results and a specific weight type
compare_wp_thresholds_to_sdss(all_results, 'pip_angular_bitwise')
plt.show()